In [1]:
# ============================================================
# Notebook    : 03c_lightgbm_aggregate.ipynb
# Description : Case A extension — Model ③c LightGBM-Aggregate
#               Static + YearGap + 5 aggregate history features.
#               Structurally identical to 03a/03b except for
#               FEATURE_COLS and input file (now reads notebook
#               04's aggregate output instead of notebook 01's).
# ============================================================


# ============================================================
# 0. Install dependencies (run once)
# ============================================================
# pip install lightgbm scikit-learn pandas numpy


# ============================================================
# 1. Common imports
# ============================================================
import json
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, f1_score, precision_recall_curve

SEED = 42
np.random.seed(SEED)


# ============================================================
# 2. Load AGGREGATE data (notebook 04 output) and the same
#    IDpol-level split used by every other model
# ============================================================
df = pd.read_csv("data/fremotor_multi_history_aggregate.csv")
print(f"[CHECK 1] df shape: {df.shape}")

with open("data/sequences/split_idpols.json", "r", encoding="utf-8") as f:
    split_ids = json.load(f)

train_idpols = set(split_ids["train"])
val_idpols   = set(split_ids["val"])
test_idpols  = set(split_ids["test"])

df_train = df[df["IDpol"].isin(train_idpols)].copy()
df_val   = df[df["IDpol"].isin(val_idpols)].copy()
df_test  = df[df["IDpol"].isin(test_idpols)].copy()

print(f"[CHECK 1] Row counts — train: {len(df_train):,}, "
      f"val: {len(df_val):,}, test: {len(df_test):,}")
print(f"[CHECK 1] Positive rates — "
      f"train: {df_train['Label'].mean()*100:.2f}%, "
      f"val: {df_val['Label'].mean()*100:.2f}%, "
      f"test: {df_test['Label'].mean()*100:.2f}%")


# ============================================================
# 3. Feature scope — static + YearGap + 5 aggregate features
#    NOTE: raw YearsSinceFirstClaim (999-sentinel version) is
#    EXCLUDED — only the log-transformed + flag pair is used
#    (see notebook 04 §6-B rationale)
# ============================================================
NUMERIC_COLS = [
    "Expo", "YearGap",
    "CumClaimCount", "ClaimRateSoFar", "YearsSinceFirstClaimLog",
]
BINARY_COLS  = ["PrevLabel", "HasPriorClaim"]
CAT_COLS     = ["Usage", "VehType", "VehPower"]
FEATURE_COLS = NUMERIC_COLS + BINARY_COLS + CAT_COLS
LABEL_COL    = "Label"

print(f"\n[CHECK 2] Feature scope (AGGREGATE — static + YearGap + 5 history feats):")
print(f"  Numeric     : {NUMERIC_COLS}")
print(f"  Binary      : {BINARY_COLS}")
print(f"  Categorical : {CAT_COLS}")

for d in (df_train, df_val, df_test):
    for col in CAT_COLS:
        d[col] = d[col].astype("category")

X_train, y_train = df_train[FEATURE_COLS].copy(), df_train[LABEL_COL].copy()
X_val,   y_val   = df_val[FEATURE_COLS].copy(),   df_val[LABEL_COL].copy()
X_test,  y_test  = df_test[FEATURE_COLS].copy(),  df_test[LABEL_COL].copy()

for col in CAT_COLS:
    all_cats = pd.api.types.union_categoricals(
        [X_train[col], X_val[col], X_test[col]]
    ).categories
    X_train[col] = X_train[col].cat.set_categories(all_cats)
    X_val[col]   = X_val[col].cat.set_categories(all_cats)
    X_test[col]  = X_test[col].cat.set_categories(all_cats)

print(f"[CHECK 2] X_train shape: {X_train.shape}")


# ============================================================
# 4. Train LightGBM — identical params/seed to 03a/03b
# ============================================================
POS_RATE = y_train.mean()
SCALE_POS_WEIGHT = (1 - POS_RATE) / POS_RATE
print(f"\n[CHECK 3] scale_pos_weight: {SCALE_POS_WEIGHT:.2f}")

train_set = lgb.Dataset(X_train, label=y_train, categorical_feature=CAT_COLS)
val_set   = lgb.Dataset(X_val,   label=y_val,   categorical_feature=CAT_COLS, reference=train_set)

params = {
    "objective": "binary",
    "metric": "auc",
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "scale_pos_weight": SCALE_POS_WEIGHT,
    "seed": SEED,
    "verbose": -1,
}

print(f"[CHECK 3] Training LightGBM-Aggregate...")
model = lgb.train(
    params,
    train_set,
    num_boost_round=500,
    valid_sets=[train_set, val_set],
    valid_names=["train", "val"],
    callbacks=[
        lgb.early_stopping(stopping_rounds=30),
        lgb.log_evaluation(period=50),
    ],
)
print(f"\n[CHECK 3] Best iteration: {model.best_iteration}")


# ============================================================
# 5. Evaluate on test set — timestep-level metrics
# ============================================================
y_pred_prob = model.predict(X_test, num_iteration=model.best_iteration)
y_pred_cls  = (y_pred_prob >= 0.5).astype(int)

auc = roc_auc_score(y_test, y_pred_prob)
f1  = f1_score(y_test, y_pred_cls)

precisions, recalls, thresholds = precision_recall_curve(y_test, y_pred_prob)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
best_idx  = np.argmax(f1_scores[:-1])

print("\n===== Test set evaluation (timestep-level) =====")
print(f"Total timestep predictions: {len(y_test):,}")
print(f"AUC-ROC          : {auc:.4f}")
print(f"F1 @ thr=0.5      : {f1:.4f}")
print(f"Best threshold    : {thresholds[best_idx]:.3f}")
print(f"Best F1           : {f1_scores[best_idx]:.4f}")
print(f"  Precision       : {precisions[best_idx]:.4f}")
print(f"  Recall          : {recalls[best_idx]:.4f}")
print(f"Positive rate in test: {y_test.mean()*100:.2f}%")


# ============================================================
# 6. Feature importance — this is the key diagnostic: does any
#    aggregate feature outrank VehPower (the dominant static
#    feature in ①②)?
# ============================================================
importance = pd.DataFrame({
    "feature": FEATURE_COLS,
    "gain": model.feature_importance(importance_type="gain"),
}).sort_values("gain", ascending=False)

print("\n===== Feature importance (gain) =====")
print(importance.to_string(index=False))


# ============================================================
# 7. Save model and predictions
# ============================================================
model.save_model("data/sequences/lightgbm_aggregate_model.txt")
print("\nSaved: data/sequences/lightgbm_aggregate_model.txt")

np.savez(
    "data/sequences/lightgbm_aggregate_test_predictions.npz",
    probs=y_pred_prob,
    labels=y_test.values,
)
print("Saved: data/sequences/lightgbm_aggregate_test_predictions.npz")


# ============================================================
# 8. Summary check
# ============================================================
print("\n===== LightGBM-Aggregate (Model ③c) Summary =====")
print(f"Feature scope       : {FEATURE_COLS}")
print(f"Train / Val / Test rows: {len(X_train):,} / {len(X_val):,} / {len(X_test):,}")
print(f"Best iteration      : {model.best_iteration}")
print(f"Test AUC-ROC        : {auc:.4f}")
print(f"Test F1 (best thr)  : {f1_scores[best_idx]:.4f}")
print("===============================================")

[CHECK 1] df shape: (364997, 16)
[CHECK 1] Row counts — train: 255,241, val: 55,001, test: 54,755
[CHECK 1] Positive rates — train: 12.66%, val: 12.63%, test: 12.79%

[CHECK 2] Feature scope (AGGREGATE — static + YearGap + 5 history feats):
  Numeric     : ['Expo', 'YearGap', 'CumClaimCount', 'ClaimRateSoFar', 'YearsSinceFirstClaimLog']
  Binary      : ['PrevLabel', 'HasPriorClaim']
  Categorical : ['Usage', 'VehType', 'VehPower']
[CHECK 2] X_train shape: (255241, 10)

[CHECK 3] scale_pos_weight: 6.90
[CHECK 3] Training LightGBM-Aggregate...
Training until validation scores don't improve for 30 rounds
[50]	train's auc: 0.799976	val's auc: 0.79456
[100]	train's auc: 0.802429	val's auc: 0.795381
Early stopping, best iteration is:
[100]	train's auc: 0.802429	val's auc: 0.795381

[CHECK 3] Best iteration: 100

===== Test set evaluation (timestep-level) =====
Total timestep predictions: 54,755
AUC-ROC          : 0.7919
F1 @ thr=0.5      : 0.3811
Best threshold    : 0.601
Best F1           :